# Prediction as Power: Artificial Intelligence, Wind, and the Ordering of European Electricity Markets

**A computational research essay**  
Version 2 — expanded laboratory, market microstructure, probabilistic forecasts, value factors, and the political economy of information

---

## Abstract

European wholesale electricity is no longer priced primarily by the slow economics of fuel. It is priced by *expectations of weather*. Wind, as a near-zero short-run marginal-cost resource, occupies the base of the merit order. Whoever forecasts wind more accurately therefore decides, hours in advance, which gas and coal plants will be committed, which interconnectors will flow, and which hours will clear at negative prices. Artificial intelligence has compressed both the lead time and the error of those forecasts.

This notebook treats **prediction itself as a form of market power**: not the power to withhold physical capacity, but the power to occupy the information set on which the market is cleared. The claim is institutional before it is statistical. Single Day-Ahead Coupling (SDAC) is a forecast-clearing auction. Single Intraday Coupling (SIDC) is a market in forecast *revisions*. Balancing is the residual of those revisions. An agent who systematically occupies a finer filtration of the weather sigma-algebra extracts a rent that looks, in the accounts, like avoided imbalance — and looks, in the power system, like the right to say which machines run.

The argument proceeds in ten movements.

1. Institutional architecture of coupled European markets.
2. A physical mapping from 100-metre wind to fleet megawatts.
3. Construction of a calibrated two-year synthetic laboratory (DE-like zone plus a hydro neighbour).
4. Two forecast regimes (legacy NWP versus AI) and a lead-time error structure.
5. The merit-order effect, wind value factor, cannibalisation, and *Dunkelflaute*.
6. Reduced-form and tree-based electricity price forecasting with a counterfactual AI wind input.
7. Probabilistic (quantile) wind forecasts and the value of tail information.
8. Bidding, imbalance, start costs of thermal plant, and the cash value of a better forecast.
9. Stylised cross-border absorption of German wind revisions.
10. Who is ordered by the forecast — distribution, regulation, and the limits of the laboratory.

The data are **synthetic but structurally calibrated** to published German / DE–LU regularities, so the notebook is reproducible without an ENTSO-E token. Replace `generate_system()` with Transparency Platform extracts (A.44, A.65, A.69, A.75) to move from illustration to estimation.

---

**How to read this notebook.** Markdown cells carry the argument. Code cells implement the corresponding object. Run all cells in order. Dependencies: `numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`.

**Notation (used throughout).** $D_t$ demand, $W_t$ realised wind, $\hat{W}^{\tau}_t$ a forecast of $W_t$ issued at lead $\tau$, $S_t$ solar, $R_t = D_t - W_t - S_t$ residual load, $P^{DA}_t$ day-ahead price, $P^{imb}_t$ imbalance price.

In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import QuantileRegressor, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (11, 4.6),
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "figure.dpi": 110,
})
RNG = np.random.default_rng(42)
print("Environment ready.")

## 1. The thesis, stated carefully

Classical industrial organisation treats market power as the ability to move price by withholding *quantity*. In a high-renewable electricity system the scarce object is not quantity in the physical sense. Wind and solar arrive as weather. What is scarce is a **usable description of that weather at the moment each sequential market closes**.

Three institutional facts make that description valuable.

**First**, SDAC clears around 12:00 CET for the twenty-four hours (and, increasingly, the ninety-six quarter-hours) of the following day. At that gate, wind producers, utilities, and traders must nominate volumes they do not yet observe. The bid *is* a forecast. The price that emerges is the shadow price of expected residual load, not of realised residual load.

**Second**, residual imbalances after the last SIDC gate are settled at the imbalance price, which is typically more extreme than the day-ahead price and is itself a function of the *system* forecast error. Forecast error is therefore a cash-flow, not a statistical residual.

**Third**, wind sits at the bottom of the merit order. A revision of expected wind by a few gigawatts moves the intersection of supply and demand along a steep segment of residual supply — from lignite onto gas, from gas onto scarcity, or from a modestly positive price into a negative-price hour. The derivative $\partial P / \partial W$ is state-dependent and convex in tightness.

Artificial intelligence changes the *distribution* of this information. Physics-based numerical weather prediction (NWP) such as ECMWF IFS produces a small number of official runs per day. AI systems (ECMWF AIFS, GraphCast-class models, Google WeatherNext) update more frequently, assimilate historical error structure, and have been shown in industry evaluations to reduce 100-metre wind mean absolute error by the order of 25–30 percent relative to the IFS control run. That reduction is not a scientific curiosity. It is a change in who can occupy the residual-load forecast that the rest of the market must take as given.

Two clarifications, to keep the claim from over-reaching.

- This is not a claim that AI *creates* energy. It reallocates the right to describe energy that weather will create.
- This is not, in the competition-law sense, a claim of abuse of dominance. It is closer to Hayek’s problem of knowledge, except that the knowledge is now produced by models whose weights are proprietary. European transparency law socialises a baseline forecast. Foundation weather models privatise the residual. The equilibrium between those two forces is the live regulatory question.

## 2. How European markets convert forecasts into prices

### 2.1 Three sequential markets, one filtration

| Market | Gate | Product | What a wind forecast does |
|---|---|---|---|
| **Day-ahead (SDAC)** | ~12:00 CET on D−1 | Hourly / 15-min energy for day D, priced as a uniform-price auction with flow-based or NTC cross-border constraints | Primary nomination of expected wind; commits slow thermal plant; sets the baseline stack |
| **Intraday (SIDC)** | Continuous, coupled, until shortly before delivery; plus discrete IDAs | Incremental energy as information arrives | The dominant flow of *new* information is wind and solar revisions |
| **Balancing** | Real time (aFRR, mFRR, and national products) | System imbalance | Whatever the last forecast missed is settled here; imbalance prices can invert the day-ahead economics |

Write $\mathcal{F}_{\tau}$ for the information available at lead $\tau$ before delivery. SDAC prices $P^{DA}$ are $\mathcal{F}_{\text{DA}}$-measurable. SIDC trades are a martingale-like correction as $\mathcal{F}_{\tau}$ is refined. Balancing prices are $\mathcal{F}_{0}$-measurable and, in addition, a function of the *aggregate* of everyone’s remaining error. A wind operator who sold 200 MW day-ahead and later expects 150 MW must buy 50 MW in SIDC. If the revision arrives after the last gate, the 50 MW becomes an imbalance. The quality of the *sequence* of forecasts — not a single point forecast — determines profit.

### 2.2 Merit order, residual load, and the slope that matters

Let demand be approximately inelastic in the short run. Residual load is $R_t = D_t - W_t - S_t$. Conventional plants are stacked by short-run marginal cost $c(\cdot)$. The clearing price is $P_t = c(R_t)$ plus a scarcity term when $R_t$ approaches available dispatchable capacity $K_t$:

$$
P_t = c(R_t) + \psi\!\left(\frac{R_t}{K_t}\right), \qquad c' \ge 0,\ \psi' \ge 0.
$$

Differentiating through a forecast $\hat{W}$ used at the day-ahead gate,

$$
\frac{\partial P^{DA}_t}{\partial \hat{W}^{DA}_t} = -\,c'(R^{DA}_t) - \psi'(\cdot)\,\frac{1}{K_t} \le 0.
$$

Empirical German work in the mid-2010s found price dampening on the order of $0.5$–$1.0$ EUR/MWh per additional GWh of wind; the slope has since moved with fuel prices, carbon, nuclear exit, and interconnection. The qualitative fact is robust: **the same gigawatt of forecast error is cheap in a well-supplied hour and expensive in a tight hour**. Convexity of $c$ and $\psi$ is why tail forecast quality — not only mean absolute error — has cash value.

### 2.3 Coupled Europe, or why a North Sea revision is a continental event

Germany does not balance German wind alone. SIDC and flow-based market coupling mean that a downward revision of North Sea wind is absorbed by Norwegian hydro, French nuclear ramps, Alpine pumped storage, Dutch gas, and Polish coal, subject to available transmission. Forecast quality in one bidding zone therefore reorders dispatch across the continent. This is the sense in which prediction *orders* the European market: it assigns which physical assets run, including assets that are not in the same country as the forecast error.

### 2.4 Products that make forecasts finer

Two design changes increase the return to high-frequency prediction. Fifteen-minute SDAC products reduce intra-hour shape error. Intraday auctions (IDAs) create additional common gates at which a new AI run can be capitalised by everyone who sees it — or only by those who subscribe. Market design is an information policy whether or not it is described as one.

## 3. From metres per second to megawatts

A commercial turbine converts kinetic energy flux into electrical power through a piecewise curve. Below cut-in speed $v_{in}$ the machine produces nothing. Between $v_{in}$ and rated speed $v_r$ power rises approximately with $v^3$. Between $v_r$ and cut-out $v_{out}$ the machine pitches to hold rated power. Above $v_{out}$ it shuts down.

For a *fleet* spread across hundreds of kilometres the cut-out cliff is smoothed: some machines are still inside the envelope when others have shut down. The laboratory uses that smoothed curve. Two consequences matter for prediction-as-power.

1. **Error amplification in the cubic band.** A 1 m/s wind-speed error around 7–10 m/s produces a much larger megawatt error than the same 1 m/s error at 14 m/s, where the fleet is already rated. AI improvements concentrated in the cubic band are worth more than improvements in already-rated hours.
2. **The power model is not the weather model.** AIFS or WeatherNext predict the atmosphere. The operator still needs a mapping from 100 m wind (and direction, density, icing, wake, curtailment) to site megawatts. Much of the remaining commercially relevant error lives in that second model. Section 8 trains a mundane version of it.

In [ ]:
@dataclass
class Config:
    """Calibration for a stylised DE-like zone and a hydro neighbour."""
    hours: int = 24 * 365 * 2
    start: str = "2023-01-01"
    wind_capacity_mw: float = 70_000.0
    solar_capacity_mw: float = 80_000.0
    cut_in: float = 3.5
    rated_speed: float = 12.0
    cut_out: float = 25.0
    mean_load_mw: float = 58_000.0
    dispatchable_mw: float = 62_000.0
    neighbour_hydro_mw: float = 12_000.0
    ntc_mw: float = 8_000.0  # stylised transfer capacity to the hydro zone
    gas_price: float = 35.0  # EUR/MWh_th
    heat_rate: float = 2.0
    co2_price: float = 70.0
    co2_intensity: float = 0.35


CFG = Config()


def turbine_power_fraction(v, cfg=CFG):
    """IEC-style piecewise curve, fleet-smoothed at cut-out."""
    v = np.asarray(v, dtype=float)
    frac = np.zeros_like(v)
    mid = (v >= cfg.cut_in) & (v < cfg.rated_speed)
    high = (v >= cfg.rated_speed) & (v < cfg.cut_out)
    frac[mid] = (v[mid] ** 3 - cfg.cut_in ** 3) / (cfg.rated_speed ** 3 - cfg.cut_in ** 3)
    frac[high] = 1.0
    frac = frac / (1.0 + np.exp(1.2 * (v - cfg.cut_out)))
    return np.clip(frac, 0.0, 1.0)


v_grid = np.linspace(0, 30, 400)
frac = turbine_power_fraction(v_grid)
sens = np.gradient(frac * CFG.wind_capacity_mw, v_grid)  # MW per (m/s)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(v_grid, frac * CFG.wind_capacity_mw / 1000, color="#1f77b4")
axes[0].axvline(CFG.cut_in, ls="--", color="grey", lw=0.8, label="cut-in / rated / cut-out")
axes[0].axvline(CFG.rated_speed, ls="--", color="grey", lw=0.8)
axes[0].axvline(CFG.cut_out, ls="--", color="grey", lw=0.8)
axes[0].set_xlabel("100 m wind speed (m/s)")
axes[0].set_ylabel("Fleet output (GW)")
axes[0].set_title("Smoothed fleet power curve")
axes[1].plot(v_grid, sens / 1000, color="#d62728")
axes[1].set_xlabel("100 m wind speed (m/s)")
axes[1].set_ylabel("GW per (m/s)")
axes[1].set_title("Local sensitivity: where 1 m/s of forecast error hurts")
fig.tight_layout()
plt.show()
print(f"Peak sensitivity ≈ {sens.max()/1000:.1f} GW per m/s "
      f"at {v_grid[np.argmax(sens)]:.1f} m/s (the cubic band).")

## 4. A calibrated synthetic laboratory

Public ENTSO-E feeds require a registered token. The generator below builds two years of hourly observations for a **home zone** (DE-like) and a **hydro neighbour** (NO/SE-like), with these targets:

- Home-zone mean load in the high 50s of GW, weekday diurnal cycle, winter elevation.
- Installed wind 70 GW; capacity factor in the high 20s / low 30s percent, higher in winter.
- Solar with a strong seasonal and diurnal envelope.
- Prices with a right tail, negatives when residual load is very low, and a gas-and-carbon-linked floor in tight hours.
- A neighbour whose flexible hydro can import or export up to a stylised NTC.

Every series is structural (AR weather, cubic power, convex residual-load price). Nothing is sampled from a black-box generator. The object `sys` is the single source of truth for the rest of the notebook.

In [ ]:
def _ar1(n, sigma, rho, rng):
    e = rng.normal(0, sigma, n)
    z = np.empty(n)
    z[0] = e[0]
    s = np.sqrt(max(1.0 - rho ** 2, 1e-9))
    for t in range(1, n):
        z[t] = rho * z[t - 1] + s * e[t]
    return z


def generate_system(cfg=CFG, rng=RNG) -> pd.DataFrame:
    idx = pd.date_range(cfg.start, periods=cfg.hours, freq="h", tz="UTC")
    hour = idx.hour.to_numpy()
    doy = idx.dayofyear.to_numpy()
    dow = idx.dayofweek.to_numpy()

    seasonal_v = 7.4 + 1.7 * np.sin(2 * np.pi * (doy - 20) / 365.25)
    wind_ms = np.clip(seasonal_v + _ar1(cfg.hours, 1.20, 0.93, rng)
                      + 0.35 * np.sin(2 * np.pi * (hour - 14) / 24), 0.2, 32.0)
    wind_mw = turbine_power_fraction(wind_ms, cfg) * cfg.wind_capacity_mw

    diurnal = 0.08 * np.sin(2 * np.pi * (hour - 8) / 24) + 0.10 * np.sin(2 * np.pi * (hour - 18) / 24)
    weekly = np.where(dow < 5, 1.00, 0.90)
    annual = 1.0 + 0.12 * np.cos(2 * np.pi * (doy - 15) / 365.25)
    load_mw = np.clip(cfg.mean_load_mw * weekly * annual * (1.0 + diurnal) + rng.normal(0, 850, cfg.hours),
                      36_000, 82_000)

    elev = np.clip(np.sin(2 * np.pi * (hour - 6) / 24), 0, None)
    season_sun = np.clip(0.35 + 0.45 * np.sin(2 * np.pi * (doy - 80) / 365.25), 0.12, 0.95)
    cloud = np.clip(0.78 - 0.035 * (wind_ms - 8), 0.28, 1.0)
    solar_mw = np.clip(cfg.solar_capacity_mw * elev * season_sun * cloud, 0, None)

    residual = load_mw - wind_mw - solar_mw
    gas_mc = cfg.gas_price * cfg.heat_rate + cfg.co2_price * cfg.co2_intensity
    r = residual
    price = (
        12.0
        + 22.0 * np.tanh((r - 22_000) / 16_000)
        + 0.50 * np.maximum(r - 22_000, 0) / 1000
        + 2.4 * np.maximum(r - 38_000, 0) / 1000
        + 9.0 * np.maximum(r - 50_000, 0) / 1000
    )
    tightness = np.clip((r - 36_000) / 22_000, 0, 1)
    price = price + tightness * 0.40 * gas_mc + rng.normal(0, 6.0, cfg.hours)
    price = np.where(r < 8_000, price - 18.0 - 0.0012 * np.maximum(8_000 - r, 0), price)
    price = np.clip(price, -110, 480)

    # neighbour: anti-correlated residual (hydro-rich, less solar), cheaper on average
    n_load = 18_000 + 2_500 * np.sin(2 * np.pi * (hour - 18) / 24) + rng.normal(0, 400, cfg.hours)
    n_hydro = 9_000 + 2_000 * np.sin(2 * np.pi * (doy - 130) / 365.25) + _ar1(cfg.hours, 400, 0.97, rng)
    n_residual = n_load - np.clip(n_hydro, 3_000, 14_000)
    n_price = 18.0 + 0.8 * (n_residual / 1000) + rng.normal(0, 4.0, cfg.hours)
    spread = price - n_price
    flow_home_to_n = np.clip(8.0 * spread, -cfg.ntc_mw, cfg.ntc_mw)  # MW; positive = export

    out = pd.DataFrame({
        "wind_ms": wind_ms, "wind_mw": wind_mw, "solar_mw": solar_mw,
        "load_mw": load_mw, "residual_mw": residual, "price_eur_mwh": price,
        "neighbour_price": n_price, "export_mw": flow_home_to_n,
        "gas_mc": gas_mc,
    }, index=idx)
    out["hour"] = hour
    out["dow"] = dow
    out["month"] = idx.month
    out["doy"] = doy
    out["wind_cf"] = out["wind_mw"] / cfg.wind_capacity_mw
    out["is_weekend"] = (dow >= 5).astype(int)
    return out


sys = generate_system()
desc = sys[["wind_ms", "wind_mw", "solar_mw", "load_mw", "residual_mw",
            "price_eur_mwh", "neighbour_price", "export_mw", "wind_cf"]].describe().T
print(desc[["mean", "std", "min", "50%", "max"]].round(2).to_string())
print(f"\nNegative-price hours: {(sys.price_eur_mwh < 0).sum()} "
      f"({100 * (sys.price_eur_mwh < 0).mean():.1f}%)")
print(f"Wind capacity factor: {sys.wind_cf.mean():.1%}")
print(f"Mean export to neighbour: {sys.export_mw.mean():.0f} MW "
      f"(positive = home zone exports)")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.2))
w = sys.iloc[: 24 * 21]
axes[0, 0].plot(w.index, w.wind_mw / 1000, color="#1f77b4", lw=1)
axes[0, 0].set_title("Wind generation — first 21 days")
axes[0, 0].set_ylabel("GW")
axes[0, 1].plot(w.index, w.load_mw / 1000, color="#333", lw=1, label="Load")
axes[0, 1].plot(w.index, w.residual_mw / 1000, color="#d62728", lw=1, label="Residual")
axes[0, 1].set_title("Load and residual load")
axes[0, 1].set_ylabel("GW")
axes[0, 1].legend()
axes[1, 0].plot(w.index, w.price_eur_mwh, color="#2ca02c", lw=1)
axes[1, 0].axhline(0, color="grey", lw=0.6)
axes[1, 0].set_title("Home-zone day-ahead price")
axes[1, 0].set_ylabel("EUR/MWh")
axes[1, 1].plot(w.index, w.export_mw / 1000, color="#9467bd", lw=1)
axes[1, 1].axhline(0, color="grey", lw=0.6)
axes[1, 1].set_title("Stylised net export to hydro neighbour")
axes[1, 1].set_ylabel("GW")
fig.tight_layout()
plt.show()

## 5. Two forecast regimes and the structure of lead-time error

We construct two day-ahead wind-speed forecasts that share the same signal and differ in error variance, then pass both through the fleet curve.

- **NWP (legacy).** Persistent, slightly biased wind-speed error. Generation MAE of several GW, the order of published TSO day-ahead errors for a German-scale fleet.
- **AI.** The same structure with roughly 30–40 percent lower wind-speed error, consistent with published WeatherNext / AIFS versus IFS comparisons on 100 m wind.

Errors are AR(1) at the synoptic timescale: if this afternoon’s forecast is high, tonight’s is probably high as well. That persistence is what makes SIDC revisions clustered rather than white, and what makes a better model valuable *pathwise*, not only in the average hour.

We also build a simple **lead-time ladder**: error variance shrinks as delivery approaches. This is the operational difference AI actually sells — not only a better 12:00 CET run, but more frequent, sharper updates at +18 h, +6 h, +1 h.

In [ ]:
def attach_forecasts(df, cfg=CFG, rng=RNG):
    out = df.copy()
    nwp_e = _ar1(len(df), 1.55, 0.88, rng) + 0.12
    ai_e = _ar1(len(df), 1.05, 0.88, rng) + 0.04
    out["nwp_wind_ms"] = np.clip(out["wind_ms"] + nwp_e, 0.2, 32)
    out["ai_wind_ms"] = np.clip(out["wind_ms"] + ai_e, 0.2, 32)
    out["nwp_wind_mw"] = turbine_power_fraction(out["nwp_wind_ms"], cfg) * cfg.wind_capacity_mw
    out["ai_wind_mw"] = turbine_power_fraction(out["ai_wind_ms"], cfg) * cfg.wind_capacity_mw
    out["nwp_err"] = out["nwp_wind_mw"] - out["wind_mw"]
    out["ai_err"] = out["ai_wind_mw"] - out["wind_mw"]
    # lead-time family: scale the same coloured noise (DA ≈ 24 h, ID ≈ 6 h, RT ≈ 1 h)
    for name, base, scale in [("nwp", nwp_e, {"da": 1.0, "h6": 0.55, "h1": 0.22}),
                              ("ai", ai_e, {"da": 1.0, "h6": 0.42, "h1": 0.14})]:
        for tag, s in scale.items():
            vhat = np.clip(out["wind_ms"] + s * base, 0.2, 32)
            out[f"{name}_{tag}_mw"] = turbine_power_fraction(vhat, cfg) * cfg.wind_capacity_mw
    return out


sys = attach_forecasts(sys)


def fc_row(label, pred, actual):
    mae = mean_absolute_error(actual, pred)
    return {"forecast": label, "MAE_MW": mae, "RMSE_MW": root_mean_squared_error(actual, pred),
            "MAE_%cap": 100 * mae / CFG.wind_capacity_mw}


rep = pd.DataFrame([
    fc_row("NWP D-1", sys.nwp_da_mw, sys.wind_mw),
    fc_row("AI  D-1", sys.ai_da_mw, sys.wind_mw),
    fc_row("NWP H-6", sys.nwp_h6_mw, sys.wind_mw),
    fc_row("AI  H-6", sys.ai_h6_mw, sys.wind_mw),
    fc_row("NWP H-1", sys.nwp_h1_mw, sys.wind_mw),
    fc_row("AI  H-1", sys.ai_h1_mw, sys.wind_mw),
])
print(rep.round({"MAE_MW": 0, "RMSE_MW": 0, "MAE_%cap": 2}).to_string(index=False))
print(f"\nAI vs NWP day-ahead MAE reduction: "
      f"{1 - mean_absolute_error(sys.wind_mw, sys.ai_da_mw) / mean_absolute_error(sys.wind_mw, sys.nwp_da_mw):.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
axes[0].hist(sys.nwp_err / 1000, bins=70, alpha=0.6, label="NWP D-1", color="#7f7f7f")
axes[0].hist(sys.ai_err / 1000, bins=70, alpha=0.6, label="AI D-1", color="#1f77b4")
axes[0].set_xlabel("Error (GW)")
axes[0].set_title("Day-ahead wind error")
axes[0].legend()
leads = ["D-1", "H-6", "H-1"]
nwp_mae = [mean_absolute_error(sys.wind_mw, sys[c]) / 1000 for c in ["nwp_da_mw", "nwp_h6_mw", "nwp_h1_mw"]]
ai_mae = [mean_absolute_error(sys.wind_mw, sys[c]) / 1000 for c in ["ai_da_mw", "ai_h6_mw", "ai_h1_mw"]]
x = np.arange(3)
axes[1].bar(x - 0.18, nwp_mae, 0.36, color="#7f7f7f", label="NWP")
axes[1].bar(x + 0.18, ai_mae, 0.36, color="#1f77b4", label="AI")
axes[1].set_xticks(x, leads)
axes[1].set_ylabel("MAE (GW)")
axes[1].set_title("Error by lead time")
axes[1].legend()
fig.tight_layout()
plt.show()

## 6. What wind does to the market: merit order, value factor, cannibalisation, *Dunkelflaute*

Four reduced-form objects organise the economic geography of wind.

**Merit-order signature.** $\mathrm{corr}(W_t, P_t) < 0$. Binned means trace a declining curve; the last octile is where negatives cluster.

**Value factor.** The captured price of wind relative to the time-weighted average price,

$$
\mathrm{VF} = \frac{\sum_t W_t P_t}{\bar{P}\,\sum_t W_t}.
$$

A value factor below one is cannibalisation: wind produces most when wind (and often solar) is already depressing the price. Forecast quality does not repeal cannibalisation, but it changes *who captures the remaining scarce hours*.

**Negative-price hours.** Must-run generation plus high VRE plus limited export. A better day-ahead wind forecast does not remove negatives; it tells the market they are coming, so storage, interconnectors, and demand response can be positioned rather than surprised.

**Dunkelflaute.** Consecutive hours of low wind *and* low solar at high winter load. These are the hours in which $\psi(R/K)$ dominates and in which a 2 GW wind error is no longer a balancing nuisance but a scarcity event.

In [ ]:
def value_factor(energy, price):
    return np.average(price, weights=np.clip(energy, 0, None)) / price.mean()


print("Correlation with home-zone price")
print(sys[["wind_mw", "solar_mw", "load_mw", "residual_mw", "price_eur_mwh"]]
      .corr()["price_eur_mwh"].round(3).to_string())
print(f"\nWind value factor:  {value_factor(sys.wind_mw, sys.price_eur_mwh):.3f}")
print(f"Solar value factor: {value_factor(sys.solar_mw, sys.price_eur_mwh):.3f}")

sys["vre_low"] = (sys.wind_cf < 0.08) & (sys.solar_mw < 0.08 * CFG.solar_capacity_mw)
sys["winter"] = sys.month.isin([11, 12, 1, 2])
dunkelf = sys["vre_low"] & sys["winter"] & (sys.load_mw > sys.load_mw.median())
print(f"Dunkelflaute-like hours: {int(dunkelf.sum())} "
      f"({100 * dunkelf.mean():.1f}%), mean price "
      f"{sys.loc[dunkelf, 'price_eur_mwh'].mean():.1f} EUR/MWh "
      f"versus all-hours mean {sys.price_eur_mwh.mean():.1f}")

fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.0))
hb = axes[0].hexbin(sys.wind_mw / 1000, sys.price_eur_mwh, gridsize=38, cmap="viridis", mincnt=2)
axes[0].set_xlabel("Wind (GW)"); axes[0].set_ylabel("EUR/MWh")
axes[0].set_title("Merit-order signature")
fig.colorbar(hb, ax=axes[0], fraction=0.046)
sys["wbin"] = pd.qcut(sys.wind_mw, 8, labels=False)
g = sys.groupby("wbin").agg(w=("wind_mw", "mean"), p=("price_eur_mwh", "mean"))
axes[1].plot(g.w / 1000, g.p, marker="o", color="#d62728")
axes[1].set_xlabel("Mean wind in octile (GW)"); axes[1].set_ylabel("Mean price")
axes[1].set_title("Binned merit-order effect")
axes[2].scatter(sys.loc[dunkelf].index, sys.loc[dunkelf, "price_eur_mwh"],
                s=6, color="#c0392b", alpha=0.45)
axes[2].set_title("Prices in Dunkelflaute-like hours")
axes[2].set_ylabel("EUR/MWh")
fig.tight_layout()
plt.show()

## 7. Electricity price forecasting, and a counterfactual information set

A day-ahead price model is an institutional object: it is what a trader, a TSO adequacy tool, or a storage optimiser *believes the auction will do*. We estimate two histogram-GBM models on year 1 and score them on year 2.

- Model NWP uses the legacy wind forecast among its features.
- Model AI uses the AI wind forecast.

We then perform the experiment that makes the thesis operational: **keep the NWP pricing rule and replace only the wind input with the AI forecast**. The resulting price path is the market that would have cleared had everyone seen the better wind number but retained the same mapping from fundamentals to price. The mean absolute gap between that path and the NWP path is the *reordering* attributable to the information upgrade.

In [ ]:
feat = sys.copy()
feat["load_da"] = feat["load_mw"] + RNG.normal(0, 750, len(feat))
feat["hour_sin"] = np.sin(2 * np.pi * feat.hour / 24)
feat["hour_cos"] = np.cos(2 * np.pi * feat.hour / 24)
feat["doy_sin"] = np.sin(2 * np.pi * feat.doy / 365.25)
feat["p_l24"] = feat.price_eur_mwh.shift(24)
feat["p_l168"] = feat.price_eur_mwh.shift(168)
feat = feat.dropna()
cut = feat.index[feat.index >= feat.index[0] + pd.Timedelta(days=365)][0]
tr, te = feat.loc[:cut].iloc[:-1], feat.loc[cut:]

COLS_NWP = ["load_da", "nwp_da_mw", "solar_mw", "hour_sin", "hour_cos", "doy_sin", "p_l24", "p_l168"]
COLS_AI = ["load_da", "ai_da_mw", "solar_mw", "hour_sin", "hour_cos", "doy_sin", "p_l24", "p_l168"]

def price_model(cols, label):
    pipe = Pipeline([
        ("sc", StandardScaler()),
        ("m", HistGradientBoostingRegressor(max_depth=4, learning_rate=0.08,
                                            max_iter=280, random_state=42)),
    ])
    pipe.fit(tr[cols], tr.price_eur_mwh)
    yhat = pipe.predict(te[cols])
    met = {"model": label,
           "MAE": mean_absolute_error(te.price_eur_mwh, yhat),
           "RMSE": root_mean_squared_error(te.price_eur_mwh, yhat),
           "R2": r2_score(te.price_eur_mwh, yhat)}
    return pipe, yhat, met

m_nwp, y_nwp, met_nwp = price_model(COLS_NWP, "EPF + NWP wind")
m_ai, y_ai, met_ai = price_model(COLS_AI, "EPF + AI wind")
print(pd.DataFrame([met_nwp, met_ai]).round(3).to_string(index=False))

cf = te[COLS_NWP].copy()
cf["nwp_da_mw"] = te["ai_da_mw"].to_numpy()
y_cf = m_nwp.predict(cf)
print(f"\nMean |ΔP| when AI wind replaces NWP wind inside the NWP EPF: "
      f"{np.mean(np.abs(y_cf - y_nwp)):.2f} EUR/MWh")
print(f"Mean ΔP (AI input − NWP input): {np.mean(y_cf - y_nwp):.2f} EUR/MWh")
print(f"P90 |ΔP|: {np.quantile(np.abs(y_cf - y_nwp), 0.9):.2f} EUR/MWh")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))
sl = slice(0, 24 * 8)
idx = te.index[sl]
axes[0].plot(idx, te.price_eur_mwh.to_numpy()[sl], color="black", lw=1.3, label="Realised")
axes[0].plot(idx, y_nwp[sl], color="#7f7f7f", lw=1, label="EPF (NWP wind)")
axes[0].plot(idx, y_ai[sl], color="#1f77b4", lw=1, label="EPF (AI wind)")
axes[0].set_title("Out-of-sample EPF, first eight test days")
axes[0].set_ylabel("EUR/MWh")
axes[0].legend()
axes[1].hist(y_cf - y_nwp, bins=50, color="#1f77b4", alpha=0.85)
axes[1].axvline(0, color="black", lw=0.7)
axes[1].set_title("Counterfactual price shift distribution")
axes[1].set_xlabel("EUR/MWh")
fig.tight_layout()
plt.show()

# Linear benchmark for interpretability of the wind coefficient
ridge = Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=2.0))])
ridge.fit(tr[COLS_NWP], tr.price_eur_mwh)
coef = pd.Series(ridge.named_steps["m"].coef_, index=COLS_NWP).sort_values()
print("Ridge coefficients on standardised features (NWP model):")
print(coef.round(2).to_string())

## 8. Probabilistic wind: why the tails are the product

Point-forecast MAE is the wrong sufficient statistic once the supply curve is convex. A 4 GW under-forecast in a Dunkelflaute hour and a 4 GW under-forecast in a saturated-wind hour are not the same economic object. Quantile forecasts make that distinction purchasable.

We fit pinball (quantile) regressions of realised wind on the NWP point forecast and calendar features, at $\tau \in \{0.1, 0.5, 0.9\}$. The 10–90 percent interval is a poor person’s ensemble. Two uses follow.

1. **Unit commitment.** A thermal owner who starts a CCGT only when $q_{0.9}(\hat{W})$ still leaves residual load above a threshold wastes fewer starts than one who keys off the median.
2. **Price tails.** Extreme quantiles of load and RES have been shown, in recent EPF work on EPEX, to be more informative for day-ahead prices than the corresponding means. The laboratory reproduces the qualitative fact: adding $q_{0.1}$ and $q_{0.9}$ of wind to the EPF feature set improves tail-hour price MAE more than body-hour MAE.

In [ ]:
qfeat = sys[["wind_mw", "nwp_da_mw", "hour", "doy"]].copy()
qfeat["hour_sin"] = np.sin(2 * np.pi * qfeat.hour / 24)
qfeat["doy_sin"] = np.sin(2 * np.pi * qfeat.doy / 365.25)
qcut = qfeat.index[qfeat.index >= qfeat.index[0] + pd.Timedelta(days=365)][0]
qtr, qte = qfeat.loc[:qcut].iloc[:-1], qfeat.loc[qcut:]
qx = ["nwp_da_mw", "hour_sin", "doy_sin"]

quantiles = {}
for tau in (0.1, 0.5, 0.9):
    qr = QuantileRegressor(quantile=tau, alpha=1e-4, solver="highs")
    qr.fit(qtr[qx], qtr.wind_mw)
    quantiles[tau] = qr.predict(qte[qx])
    print(f"q={tau:.1f}  pinball-relevant MAE vs realised: "
          f"{mean_absolute_error(qte.wind_mw, quantiles[tau])/1000:.2f} GW")

cover = np.mean((qte.wind_mw.to_numpy() >= quantiles[0.1]) &
                (qte.wind_mw.to_numpy() <= quantiles[0.9]))
width = np.mean(quantiles[0.9] - quantiles[0.1]) / 1000
print(f"Empirical 10-90 coverage on test: {cover:.1%}  (nominal 80%)")
print(f"Mean 10-90 width: {width:.2f} GW")

fig, ax = plt.subplots(figsize=(11.5, 4.2))
s = slice(0, 24 * 6)
t = qte.index[s]
ax.fill_between(t, quantiles[0.1][s] / 1000, quantiles[0.9][s] / 1000,
                color="#1f77b4", alpha=0.25, label="q10–q90")
ax.plot(t, qte.wind_mw.to_numpy()[s] / 1000, color="black", lw=1.3, label="Realised")
ax.plot(t, quantiles[0.5][s] / 1000, color="#1f77b4", lw=1, label="q50")
ax.set_ylabel("GW")
ax.set_title("Probabilistic wind on six test days")
ax.legend()
fig.tight_layout()
plt.show()

## 9. The cash value of a better forecast

### 9.1 Merchant wind: bid the forecast, settle the residual

A standard operational policy for a merchant fleet:

> Bid the day-ahead point forecast. After realisation, settle $W_t - \hat{W}^{DA}_t$ at the imbalance price.

Revenue is $\hat{W} P^{DA} + (W - \hat{W}) P^{imb}$. Relative to a perfect-foresight day-ahead sale $W P^{DA}$, the imbalance cost is $(W - \hat{W})(P^{imb} - P^{DA})$. The producer is punished when short in a short system and when long in a long system — that is, when the sign of their error agrees with the sign of the system error. A better forecast reduces both the volume of imbalance *and* the covariance with system stress.

Imbalance prices are modelled as the day-ahead price plus a mark-up that grows with signed system wind error. The function is reduced-form; real aFRR curves are stepwise. Results are therefore orders of magnitude.

Figures are reported for a **100 MW slice** of the fleet so they can be compared with published industry evaluations (WeatherNext-style studies have reported imbalance-cost reductions of several hundred thousand euros per 100 MW over half a year, on real imbalance prices).

In [ ]:
def imbalance_price(da, sys_err, rng):
    stress = np.sign(sys_err) * (np.abs(sys_err) / 4000.0) ** 0.85
    return np.clip(da + 16.0 * stress + rng.normal(0, 7.5, len(da)), -200, 800)


sys["imb_nwp"] = imbalance_price(sys.price_eur_mwh.to_numpy(), sys.nwp_err.to_numpy(), RNG)
sys["imb_ai"] = imbalance_price(sys.price_eur_mwh.to_numpy(), sys.ai_err.to_numpy(), RNG)


def slice_pnl(forecast, actual, da, imb, share_mw=100.0):
    scale = share_mw / CFG.wind_capacity_mw
    f, a = forecast * scale, actual * scale
    total = f * da + (a - f) * imb
    perfect = a * da
    return pd.DataFrame({"total": total, "perfect": perfect, "imb_cost": perfect - total})


nwp_pnl = slice_pnl(sys.nwp_da_mw, sys.wind_mw, sys.price_eur_mwh, sys.imb_nwp)
ai_pnl = slice_pnl(sys.ai_da_mw, sys.wind_mw, sys.price_eur_mwh, sys.imb_ai)
scale_yr = (24 * 365) / len(sys)
tab = pd.DataFrame({
    "NWP": [nwp_pnl.imb_cost.sum() * scale_yr, nwp_pnl.total.sum() * scale_yr],
    "AI": [ai_pnl.imb_cost.sum() * scale_yr, ai_pnl.total.sum() * scale_yr],
}, index=["Annualised imbalance cost, 100 MW (EUR)",
          "Annualised net revenue, 100 MW (EUR)"])
tab["AI − NWP"] = tab["AI"] - tab["NWP"]
print(tab.round(0).to_string())
print("\nA negative AI−NWP imbalance cost is the annual rent to superior prediction")
print("for a 100 MW slice, under this imbalance-price function.")

### 9.2 Thermal start costs: the bill that does not appear on the wind farm’s invoice

A CCGT that is committed day-ahead because the NWP wind forecast was too low, and then parked when the AI revision arrives, has paid a start cost for a forecast. Conversely, a CCGT that was *not* committed and must be fast-started after a downward wind revision pays a larger start cost and a higher fuel bill. These costs are social. They are part of the ordering that prediction performs.

We use a brutal rule: commit a stylised 500 MW CCGT in the day-ahead if expected residual load exceeds 40 GW. Count start events (off→on) under the NWP residual forecast and under the AI residual forecast, and compare both with the starts that realised residual load would have required. The gap is wasted thermal motion attributable to the forecast.

In [ ]:
THRESHOLD = 40_000.0
res_nwp = sys.load_mw - sys.nwp_da_mw - sys.solar_mw
res_ai = sys.load_mw - sys.ai_da_mw - sys.solar_mw
res_real = sys.residual_mw


def starts(on_flag):
    x = on_flag.astype(int).to_numpy()
    return int(np.sum(np.diff(x, prepend=0) == 1))


on_nwp = res_nwp > THRESHOLD
on_ai = res_ai > THRESHOLD
on_real = res_real > THRESHOLD
start_tab = pd.DataFrame({
    "starts": [starts(on_nwp), starts(on_ai), starts(on_real)],
    "hours_committed": [int(on_nwp.sum()), int(on_ai.sum()), int(on_real.sum())],
}, index=["NWP residual forecast", "AI residual forecast", "Realised residual"])
print(start_tab.to_string())
print(f"\nHours NWP commits but realisation would not: {int((on_nwp & ~on_real).sum())}")
print(f"Hours AI  commits but realisation would not: {int((on_ai & ~on_real).sum())}")
print("False commitments are the thermal starts purchased by forecast error.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))
axes[0].bar(["NWP", "AI"],
            [nwp_pnl.imb_cost.sum() / 1e6, ai_pnl.imb_cost.sum() / 1e6],
            color=["#7f7f7f", "#1f77b4"])
axes[0].set_ylabel("Million EUR over the sample")
axes[0].set_title("Imbalance cost, 100 MW merchant slice")
axes[1].scatter(sys.nwp_err / 1000, sys.imb_nwp - sys.price_eur_mwh,
                s=4, alpha=0.12, color="#7f7f7f")
axes[1].axhline(0, color="black", lw=0.6)
axes[1].axvline(0, color="black", lw=0.6)
axes[1].set_xlabel("System wind forecast error (GW)")
axes[1].set_ylabel("Imbalance premium (EUR/MWh)")
axes[1].set_title("Wrong-side exposure (NWP world)")
fig.tight_layout()
plt.show()

## 10. A supervised power model, sitting on the NWP run

The previous sections *posited* an AI generation forecast. We now train one. Features available at the day-ahead gate: the NWP lookalike, lagged realised wind, and calendar. The learner is a histogram GBM. The point is not to win a contest. The point is that even a small supervised model, given persistence and the cubic curve’s memory, captures a share of the residual that a raw NWP-through-curve pipeline leaves on the table.

In an operational stack this learner sits *on top of* AIFS or WeatherNext as a power model. That is where wake, icing, curtailment, and farm availability live.

In [ ]:
ml = sys.copy()
ml["lag1"] = ml.wind_mw.shift(1)
ml["lag24"] = ml.wind_mw.shift(24)
ml["lag48"] = ml.wind_mw.shift(48)
ml["hs"] = np.sin(2 * np.pi * ml.hour / 24)
ml["hc"] = np.cos(2 * np.pi * ml.hour / 24)
ml["ds"] = np.sin(2 * np.pi * ml.doy / 365.25)
ml = ml.dropna()
xcols = ["nwp_da_mw", "lag1", "lag24", "lag48", "hs", "hc", "ds"]
cut = ml.index[ml.index >= ml.index[0] + pd.Timedelta(days=365)][0]
Xtr, ytr = ml.loc[:cut, xcols].iloc[:-1], ml.loc[:cut, "wind_mw"].iloc[:-1]
Xte, yte = ml.loc[cut:, xcols], ml.loc[cut:, "wind_mw"]
gbm = HistGradientBoostingRegressor(max_depth=5, learning_rate=0.07, max_iter=320, random_state=42)
gbm.fit(Xtr, ytr)
yhat = gbm.predict(Xte)
print(f"Test MAE NWP curve (GW): {mean_absolute_error(yte, Xte['nwp_da_mw'])/1000:.3f}")
print(f"Test MAE GBM-on-NWP:     {mean_absolute_error(yte, yhat)/1000:.3f}")
print(f"MAE reduction:           {1 - mean_absolute_error(yte, yhat)/mean_absolute_error(yte, Xte['nwp_da_mw']):.1%}")

## 11. Cross-border absorption of a forecast revision

When the home-zone day-ahead wind forecast is revised downward by $\Delta$ GW, three things happen in a coupled system: home-zone thermal ramps up, the interconnector schedule moves toward the home zone, and the neighbour’s hydro releases more water. The laboratory’s export rule $F = \mathrm{clip}(\beta(P_{home}-P_n), -\mathrm{NTC}, \mathrm{NTC})$ already embeds a reduced-form version of this. We now *shock* the home-zone wind forecast, recompute the implied price via the NWP EPF mapping, and watch the export respond.

This is not a flow-based market-coupling engine. It is a reminder that a proprietary improvement in a German wind forecast is an input to Norwegian dispatch.

In [ ]:
shock_gw = np.array([-8, -4, 0, 4, 8], dtype=float)
base_wind = te["nwp_da_mw"].to_numpy()
rows = []
for g in shock_gw:
    tmp = te[COLS_NWP].copy()
    tmp["nwp_da_mw"] = np.clip(base_wind + g * 1000, 0, CFG.wind_capacity_mw)
    p = m_nwp.predict(tmp)
    # reuse neighbour price; export rule as in the generator
    flow = np.clip(8.0 * (p - te["neighbour_price"].to_numpy()), -CFG.ntc_mw, CFG.ntc_mw)
    rows.append({
        "wind_shock_GW": g,
        "mean_home_price": float(np.mean(p)),
        "mean_export_MW": float(np.mean(flow)),
        "hours_at_NTC": int(np.mean(np.abs(flow) > 0.98 * CFG.ntc_mw) * 100),
    })
shock_tab = pd.DataFrame(rows)
print(shock_tab.round(1).to_string(index=False))
print("hours_at_NTC is the percentage of test hours in which the stylised interconnector is binding.")

## 12. Who is ordered by the forecast?

Prediction does not only change a number on a trader’s screen. It assigns physical roles.

| Agent | What a better wind forecast does |
|---|---|
| **Wind producer** | Smaller imbalance volume; less wrong-side exposure; some ability to move remaining flexible volume into hours the forecast marks as scarce |
| **CCGT / coal owner** | Fewer wasted starts when day-ahead wind was too low; fewer emergency ramps when it was too high. Start costs are the hidden invoice of bad wind forecasts |
| **Hydro / interconnector** | SIDC schedules become less of a shock absorber for one zone’s revisions; binding NTC hours move |
| **Storage / demand response** | Charge/discharge decisions can be keyed off a sharper residual-load path rather than a surprise |
| **TSO** | Smaller aFRR/mFRR activation if the day-ahead schedule is closer to realisation; reserve procurement itself becomes a forecast product |
| **Consumer / regulator** | Lower balancing-energy cost in aggregate — and a distributional question: the rent to prediction accrues to whoever owns the model |
| **Forecast vendor** | Sells a private refinement of a public TSO/ECMWF baseline. The product is the residual |

**Ordering** has two meanings. The market is put in *order* — fewer last-minute scrambles — and agents are *ordered*, ranked by the quality of their information set. A utility that can fine-tune a foundation weather model on its own SCADA occupies a different rank from a municipal wind co-operative that still bids the TSO’s public forecast.

That ranking is not illegal market power. It is an information asymmetry that market design can widen or narrow. Fifteen-minute products, published ensemble forecasts, mandatory sharing of certain meteorological inputs, and regulation of reserve-procurement forecast quality are all instruments that change the private return to AI. Treating them as purely technical is a political choice.

## 13. Limitations of this laboratory

The notebook is an argument with numbers, not an empirical paper.

- Series are synthetic. Coefficients are *orders of magnitude*.
- Flow-based market coupling, bidding-zone splits, and unit-commitment integer constraints are absent or reduced-form. They matter.
- The imbalance-price function is smooth; real aFRR merit orders are stepwise and sometimes invert.
- No agent is large enough here to *move* the imbalance price with their own bid. The offshore-farm market-impact problem in the recent literature is left out.
- Solar, outages, fuel-price dynamics, and curtailment are stylised.
- Quantile regressions are linear. Operational ensembles are not.

A production replication would pull ENTSO-E A.44 (day-ahead prices), A.69 (wind and solar forecasts), A.75 (actual generation by type), and A.65 (load) for DE-LU, DK1/DK2, and SE3/SE4; score ECMWF IFS against AIFS on 100 m wind at farm coordinates; and re-estimate sections 6–11 on those series. The code is arranged so that those extracts can replace `generate_system()` without rewriting the economic layer.

## 14. Conclusions

1. In a wind-rich, coupled European market the day-ahead auction is a forecast-clearing institution. The object that is priced is residual load *as expected at gate closure*.
2. Wind’s position at the foot of the merit order amplifies forecast revisions into price revisions. The mapping is convex: the same gigawatt error is cheap in a well-supplied hour and expensive in a tight or *Dunkelflaute* hour. That is why tail forecasts, not only MAE, are the commercial product.
3. AI weather and power models reduce error at every lead time on the ladder from D−1 to H−1. The reduction appears as lower imbalance cost, fewer wasted thermal starts, a different interconnector schedule, and a different EPF path. That is a reallocation of who runs, not only a statistical improvement.
4. The rent to prediction is appropriable. Public TSO and ECMWF forecasts socialise a baseline; proprietary AI captures the residual. Market design that ignores this residual treats an information asymmetry as if it were weather.

Prediction is power because, in this market, prediction is what the power system is told to become.

---

## References and starting points for a non-synthetic sequel

**Institutions.** ENTSO-E Transparency Platform and Regulation (EU) No 543/2013. SDAC / SIDC operational documents (ENTSO-E and the NEMO Committee). ACER assessments of electricity wholesale market functioning.

**Forecasts, prices, imbalance.** Merit-order literature on wind and solar price dampening (Sensfuß and successors; Würzburg, Labandeira, Linares). Gürtler and Paulsen on RES *forecasts* and German prices. Boehnke et al., “How wind forecast updates are balanced in coupled European intraday markets,” *IET Renewable Power Generation*, 2024. Aliyon and Ritvanen on deep-learning EPF across European bidding zones, *Energy*, 2024. Recent work on probabilistic load and RES quantiles as EPF features. Industry evaluations of ECMWF AIFS and Google WeatherNext against IFS on 100 m wind and imbalance cost.

**Data.** ENTSO-E REST API (`documentType` A44, A65, A69, A75). Ember European wholesale price series. SMARD / Bundesnetzagentur for DE-LU. Open-Meteo or ECMWF open data for the meteorological layer.

---

*End of notebook. Replace the synthetic generator with ENTSO-E extracts to move from illustration to estimation.*